In [2]:
import os
import subprocess

In [ ]:
configs_dir = 'configs'

model_config= f'{configs_dir}/config.json'
tokenizer_config = f'{configs_dir}/tokenizer_configs/config.json'
model_names = ['Qwen3.5-27B']

#### infer (serving)

In [ ]:
# start server

for model_name in model_names:
    args = [
        'python', 'infer_serving.py',
        '--engine', 'vllm',
        '--model_name', model_name,
        '--model_config', model_config,
        '--tokenizer_config', tokenizer_config
    ]

process = subprocess.Popen(
    args,
    env=os.environ.copy(),
    # capture_output=True,      
    text=True,                    # returns strings, not bytes.
    cwd=os.getcwd()               # If not specified, cwd will be equal to the laptop file folder 
                                  # (that is, from where the subprocess is launched)
)

print(f'llm server PID: {process.pid}')
PID = process.pid


In [ ]:
# request generate

import requests

url = "http://127.0.0.1:8000/llm_generate"

gen_request = {
    "conv_dataset": f'conv_dataset.jsonl',
    "output_dir": f'res',
    "batch_size": 9999,
    "tokenizer_params": {}
}

response = requests.post(url, json=gen_request)

if response.status_code == 200:
    result = response.json()
    print(result)
else:
    print(f"Error: {response.status_code}, {response.text}")

In [ ]:
# request generate (curl)

# curl -X POST "http://127.0.0.1:8000/llm_generate" \
#      -H "Content-Type: application/json" \
#      -d '{
#            "conv_dataset": "/path/to/dataset.jsonl",
#            "output_dir": "/path/to/output",
#            "batch_size": 16,
#            "tokenizer_params": {
#              "padding": true,
#              "truncation": true
#            }
#          }'

In [ ]:
# stopping the server via endpoint
response = requests.get("http://127.0.0.1:8000/shutdown")
print(response.text)

# stopping the server via a process var
# process.terminate()  # отправляет SIGTERM
# process.wait()       # ждем завершения

# !kill -9 {PID}

#### infer (no serving)

In [ ]:
# all processes are run in a separate thread to avoid blocking Jupyter.

import threading

script = 'infer.py'
chat_config = 'chat_configs/config.yml'

data_dir = '../../data/passages/'
jobs = {
    '--conv_dataset': [data_dir + d for d in [
        'infer_dataset_1.json', 
        'infer_dataset_2.json', 
        'infer_dataset_3.json']],
    '--out_file': [data_dir + d for d in [
        'infer_dataset_1_res.jsonl', 
        'infer_dataset_2_res.jsonl', 
        'infer_dataset_3_res.jsonl']]
}

script_args = {
    '--engine': 'vllm',             # 'tensorrtllm'
    '--model_name': model_name,
    '--model_config': model_config,
    '--chat_config': chat_config,
    '--conv_dataset': None,
    '--out_file': None,
    '--batch_size': '9999',    
}

procs = []
stop = False

def run_jobs():
    for args in zip(*jobs.values()):
        if stop:
            return
        script_args.update(zip(jobs.keys(), args))
        script_args_ = [kv for arg in script_args.items() for kv in arg]
        args = ['python', script] + script_args_    # 'python', 'infer.py'   # 'python', '-m', 'infer'
        print(args)
        
        proc = subprocess.run(
            args,
            env=os.environ.copy(),
            # capture_output=True,     
            text=True,                  
            cwd=os.getcwd()                
        )
        procs.append(proc)    
        # print("STDOUT:", proc.stdout)
        # print("STDERR:", proc.stderr)
        # print("Return code:", proc.returncode)


thread = threading.Thread(target=run_jobs)
thread.start()


#### infer (openai)

In [ ]:
from openai import AsyncOpenAI
import asyncio
from functools import partial
import nlplib.utils.io as io


ip_addr, port = '172.22.100.48', '8000'
base_url = f"http://{ip_addr}:{port}/v1"

client = AsyncOpenAI(
    base_url=base_url,
    api_key="not-needed",
    timeout=120.0  # Таймаут на один запрос (секунды)
)

In [ ]:
# в Jupyter можно вызывать await потому что у Jupyter (IPython) есть top-level await, то есть Event loop
# который создается ядром Jupyter при старте

print(asyncio.get_event_loop())

In [ ]:
# Ограничитель параллельных запросов (защита от перегрузки сети/vLLM)
# vLLM сам батчит запросы, но ОС/сеть могут не выдержать 1000 одновременных соединений

MAX_CONCURRENCY = 3  
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

In [ ]:
from nlplib.infer.openai_inferer import ThreadedInferer, on_done

inferer = ThreadedInferer(max_concurrent=3)

In [ ]:
for subset, convs in subset_convs.items():
    on_done_ = partial(on_done, out_file=f'infer_out_{subset}.json')
    inferer.enqueue(convs, max_seconds=12000, on_done_callback=on_done_)

In [ ]:
inferer.resume()

In [ ]:
inferer.stop()

In [ ]:
results = inferer.get_results()

In [ ]:
for res, (i, _) in zip(results_remaining, convs_remaining):
    if not isinstance(res, str):
        results[i] = res 

In [ ]:
convs_remaining = [(i, c) for i, c in enumerate(convs) if isinstance(results[i], str)]
print(len(convs_remaining))

convs_remaining_ = [c[1] for c in convs_remaining]

In [ ]:
compls = [res.choices[0].message.content if not isinstance(res, str) else None for res in results]
io.write_json(compls, 'compls_chunk.json')

In [ ]:
for res in results[:3]:
    for ch in res.choices:
        print(ch.message.content)
    print('\n' + '-'*50 + '\n')